In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"

In [5]:
import os
import torch
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [6]:
from loading_weights import gpt as model

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
model = model.to(device)

In [9]:
from dataloaders_for_classification_pretraining import train_dataloader as train_loader, validation_dataloader as val_loader, test_dataloader as test_loader


In [10]:
for param in model.parameters():
    param.requires_grad = False

In [11]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [12]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
in_features=1280,
out_features=num_classes
)

In [13]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [14]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

Inputs: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])


In [15]:
with torch.no_grad():
    outputs = model(inputs.to(device))
    print("Outputs:\n", outputs)
    print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens,num_classes)

Outputs:
 tensor([[[ 0.3659,  0.3175],
         [ 0.0619,  0.0078],
         [ 0.0955, -0.5572],
         [ 0.1592,  0.6741]]])
Outputs dimensions: torch.Size([1, 4, 2])


In [16]:
# To extract the last output token, illustrated in figure 6.11, from the output tensor, we
# use the following code:
print("Last output token:", outputs[:, -1, :])

Last output token: tensor([[0.1592, 0.6741]])


In [22]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i < num_batches:
                loss = calc_loss_batch(input_batch, target_batch, model, device)
                total_loss += loss.item()
            else:
                break
    return total_loss / num_batches

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

In [21]:
print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 0.727


NameError: name 'val_loss' is not defined

In [ ]:
def train_classifier(model, train_loader, val_loader, optimizer, device,
    num_epochs, eval_freq, eval_iter, tokenizer):
    # Initialize lists to track losses
    train_losses, val_losses = [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            examples_seen += input_batch.shape[0]
            global_step += 1

            # Only evaluate at the specified frequency
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)

                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                    f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

    return train_losses, val_losses, examples_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device,
        num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
import time
import tiktoken
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR

tokenizer = tiktoken.get_encoding('gpt2')
start_time = time.time()
torch.manual_seed(123)

# Improved optimizer settings
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=5)

num_epochs = 6

train_losses, val_losses, examples_seen = train_classifier(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
    tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
model.eval()

In [ ]:
from function_to_classify_text import classify_review

In [ ]:
text = "congrets you won a prize money of 20k."

In [ ]:
print(classify_review(text, model, tokenizer, device, max_length=256))